# Build Measured Intrinsic Wavefront (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-04-28
**Last Modified:** 2026-04-28
**Status:** Draft
**Keywords:** AOS, intrinsic, double-Zernike, FAM, focal plane

## Description

Build an empirical estimate of the focal-plane intrinsic Zernike map
("measured intrinsic") from FAM donut data, by fitting and subtracting
a chosen sparse subset of Double-Zernike modes (focal `k` × pupil `j`).

Key functionality:
1. Load donut + visit-info parquet pair, filter by day_obs / elevation /
   rotator angle / seq_num.
2. Per visit fit the chosen `(k, j)` DZ modes to `data – tabulated_intrinsic`.
3. Subtract the DZ contribution (only); take the median residual on a
   focal-plane grid per pupil j → iter-1 measured intrinsic.
4. Iterate using iter-1 as the new tabulated intrinsic → iter-2.
5. Compare original / iter-1 / iter-2 / tabulated intrinsic per pupil j
   on a common 5–95 % colour scale.
6. Save iter-2 DZ fits and the measured intrinsic map.

**Output:** parquet (or FITS / HDF5 — see § 8) with `(thx, thy)` in OCS
and CCS plus median `zk` per pupil Noll index, plus a parquet with the
iter-2 per-visit DZ fit coefficients.

**Based on:** `code/measured_intrinsic.py`, `code/dz_fitting.py`,
`code/intrinsics_lib.py`, the `run_pipeline` mktable / fit / plots flow.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-28 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Z4 Height Correction (precompute)](#z4height)
6. [DZ Fit + Iteration](#analysis)
7. [Comparison Plots — 4-panel per pupil j (2x2)](#plots)
8. [Per-DZ-term Tracking (Iter-1 vs Iter-2)](#tracking)
9. [Per-visit Residual Validation](#validation)
10. [Output Tables](#output)
11. [Output Format Options](#format)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Input -----
# Path to the donut parquet file.  The matching visits sidecar
# `{stem}_visits.parquet` is loaded automatically.
donut_parquet = 'output/aos_fam_danish_v1_triplets_20260315_20260409.parquet'

# Coordinate system for the fit and grid (FAM triplets are taken near
# rotator angle = 0, so OCS and CCS are essentially equivalent).
coord_sys = 'OCS'

# ----- Filters (None = no constraint) -----
day_obs_min   = 20260315
day_obs_max   = 20260317
alt_min_deg   = None        # elevation in degrees
alt_max_deg   = None
rot_min_deg   = None        # rotator_angle in degrees (None = no cut)
rot_max_deg   = None
seq_num_min   = None
seq_num_max   = None

# ----- Per-visit quality cuts (matches run_pipeline fit step) -----
min_donuts        = 500     # require at least this many donuts per visit
bad_fit_threshold = 2.0     # flag bad if any |coeff| > this (μm)

# ----- DZ removal spec -----
# Sparse spec: keys are focal Noll k, values are pupil Noll j to fit.
# Default = the user-specified set:
#   k=1 : j = 4..26 except 20, 21      (pentafoil_x/y omitted)
#   k=2 : j = 4..17
#   k=3 : j = 4..17
#   k=4 : j = 4..10
#   k=5 : j = 4..10
#   k=6 : j = 4..10
removal_spec = {
    1: [j for j in range(4, 27) if j not in (20, 21)],
    2: list(range(4, 18)),
    3: list(range(4, 18)),
    4: list(range(4, 11)),
    5: list(range(4, 11)),
    6: list(range(4, 11)),
}

# ----- Iteration -----
n_iter = 2

# ----- Focal-plane grid -----
n_bins = 73                  # 18*4 + 1 trio-style binning
fp_radius_basis = 1.75       # field radius for Z basis normalization
fp_radius_grid  = 1.8        # grid extent for binned medians

# ----- Z4 CCD-height correction -----
# When `height_map_fits` is set, Z4 is corrected per donut BEFORE the
# DZ fits and the binning:
#   - measured Z4: subtract Z4hgt        = 15 μm/mm × height(fpx, fpy)
#   - tabulated Z4: subtract Z4hgt_transpose
# `Z4hgt_transpose` evaluates the height at the per-CCD (x<->y)
# transpose of (fpx, fpy), to match the bug in the pipeline's
# per-CCD intrinsic Zernike calculation.  Set
# intrinsic_transpose_bug=False once the upstream tabulation is fixed.
height_map_fits          = 'output/LSST_FP_cold_b_measurement_4col_bysurface.fits'
height_to_z4_factor      = None   # None -> ccd_height.HEIGHT_TO_Z4_UM_PER_MM
intrinsic_transpose_bug  = True   # use per-CCD transposed Z4hgt for tabulated

# ----- Per-visit residual validation -----
# Histograms are cheap; the residual color maps are slow because
# `binned_statistic_2d` is called per visit per j.  Default to
# histograms only.
make_residual_maps   = False
residual_j_range     = range(4, 20)     # pupil j = 4..19 (16 panels = 4x4 fully filled)
residual_hist_range  = (-1.0, 1.0)      # μm
residual_n_hist_bins = 60
residual_map_n_bins  = 16               # coarse, like the movie binning
residual_map_fp_radius = 1.8
# Per-Zernike σ / σ_MAD vs visit summary pages: list of
# pupil j to make individual time-series plots for (in
# addition to the pooled σ over residual_j_range).
per_zernike_sigma_j = range(4, 9)        # j = 4..8

# ----- Output -----
output_dir = None            # None -> derive from donut stem + filter tag
output_format = 'parquet'    # parquet (recommended)

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.table import QTable

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

sys.path.insert(0, 'code')
from dz_fitting import derive_noll_indices
from measured_intrinsic import (
    expand_removal_spec,
    apply_visit_filters,
    bin_median_focal,
    interpolate_grid_at_donuts,
    build_measured_intrinsic,
    assemble_intrinsic_table,
    save_dz_fits,
)
# CCD focal-plane height map helpers (for the Z4 correction).  These
# imports pull in lsst.obs.lsst, sklearn, and astropy.io — they only
# work on RSP / inside the LSST conda env.
try:
    from ccd_height import (
        compute_fp_coords,
        make_metrology_table,
        get_height_interpolator,
        height_to_z4,
        transpose_around_ccd_centers,
        HEIGHT_TO_Z4_UM_PER_MM,
    )
    _ccd_height_ok = True
except Exception as _e:
    print(f'(ccd_height import skipped: {type(_e).__name__}: {_e})')
    _ccd_height_ok = False

print('Loaded measured_intrinsic helpers.')

<a id='functions'></a>
## 3. Helper Functions

In [ ]:
def filter_donuts_by_visits(donut_df, visits_kept):
    """Return rows of donut_df whose (day_obs, seq_num) appears in visits_kept."""
    keep_keys = set(zip(
        np.asarray(visits_kept['day_obs']).tolist(),
        np.asarray(visits_kept['seq_num']).tolist(),
    ))
    keys = list(zip(donut_df['day_obs'].tolist(),
                    donut_df['seq_num'].tolist()))
    mask = np.array([k in keep_keys for k in keys])
    return donut_df.loc[mask].reset_index(drop=True)


def common_color_scale(grids, plo=5.0, phi=95.0):
    """Return (vmin, vmax) covering the plo-phi percentile of *all* grids
    (skip NaNs; ignore empty grids)."""
    pooled = np.concatenate(
        [g.ravel()[~np.isnan(g.ravel())] for g in grids if g is not None
         and np.any(np.isfinite(g))])
    if pooled.size == 0:
        return -1.0, 1.0
    return tuple(np.nanpercentile(pooled, [plo, phi]))


def output_dir_default(donut_parquet, coord_sys, day_obs_min, day_obs_max,
                       seq_num_min, seq_num_max,
                       alt_min_deg, alt_max_deg,
                       rot_min_deg, rot_max_deg):
    """Build a self-describing output subdirectory name."""
    stem = Path(donut_parquet).stem
    parts = [stem, f'measured_intrinsic_{coord_sys}']
    if day_obs_min or day_obs_max:
        parts.append(f'day_{day_obs_min or ""}_{day_obs_max or ""}')
    if seq_num_min or seq_num_max:
        parts.append(f'seq_{seq_num_min or ""}_{seq_num_max or ""}')
    if alt_min_deg is not None or alt_max_deg is not None:
        parts.append(f'alt_{alt_min_deg or ""}_{alt_max_deg or ""}')
    if rot_min_deg is not None or rot_max_deg is not None:
        parts.append(f'rot_{rot_min_deg or ""}_{rot_max_deg or ""}')
    return Path('output') / '_'.join(parts)

<a id='data'></a>
## 4. Data Access

In [ ]:
donut_path = Path(donut_parquet)
visits_path = donut_path.parent / f'{donut_path.stem}_visits.parquet'
print(f'Donuts:  {donut_path}')
print(f'Visits:  {visits_path}')
assert donut_path.exists(), f'missing {donut_path}'
assert visits_path.exists(), f'missing {visits_path}'

# Load whole donut table (uses pyarrow under the hood).  For very large
# inputs the streaming variant in measured_intrinsic.fit_dz_modes_streaming
# can be substituted; for now we keep it in-memory because the iterative
# build_measured_intrinsic call needs random access by visit.
donut_full = pd.read_parquet(donut_path)
visits_full = QTable.read(str(visits_path), format='parquet')
print(f'Loaded {len(donut_full)} donuts, {len(visits_full)} visits')

# Apply visit-level filters
visits_kept = apply_visit_filters(
    visits_full,
    day_obs_min=day_obs_min, day_obs_max=day_obs_max,
    alt_min_deg=alt_min_deg, alt_max_deg=alt_max_deg,
    rotator_min_deg=rot_min_deg, rotator_max_deg=rot_max_deg,
    seq_num_min=seq_num_min, seq_num_max=seq_num_max,
)
print(f'Visits after filters: {len(visits_kept)}/{len(visits_full)}')

donut_df = filter_donuts_by_visits(donut_full, visits_kept)
print(f'Donuts after filters: {len(donut_df)}/{len(donut_full)}')

# Derive Noll indices from the data
nZk = np.stack(donut_df[f'zk_{coord_sys}'].values).shape[1]
noll_arr = (np.array(visits_kept['nollIndices'][0])
            if 'nollIndices' in visits_kept.colnames else None)
iZs, iZidx = derive_noll_indices(nZk, noll_arr)
print(f'Pupil Noll indices ({len(iZs)}): {iZs}')

pairs, by_pupil, by_focal = expand_removal_spec(removal_spec)
print(f'\nDZ removal spec: {len(pairs)} (k,j) modes')
for k, js in by_focal.items():
    print(f'  k={k}: j={js[0]}..{js[-1]} ({len(js)} terms)')

<a id='z4height'></a>
## 5. Z4 Height Correction (precompute)

Compute the per-donut Z4 contribution from the focal-plane height map
**before** any DZ fitting or binning:

* `Z4hgt`           = `15 μm/mm × height(fpx, fpy)` — for the measured data.
* `Z4hgt_transpose` = `15 μm/mm × height(fpx_swap, fpy_swap)` where
  `(fpx_swap, fpy_swap)` is the per-CCD x↔y transpose around the
  detector center — for the *tabulated* intrinsic, mirroring the bug
  in the pipeline's per-CCD intrinsic Zernike calculation.

If the metrology FITS file is unavailable or the LSST stack isn't
loaded, the corrections are skipped.  When `intrinsic_transpose_bug` is
False the tabulated correction uses `Z4hgt` directly (no transpose).

In [ ]:
data_offset = None
intrinsic_offset = None
Z4hgt = None
Z4hgt_intrinsic = None

_z4_can_correct = (_ccd_height_ok and height_map_fits
                   and Path(height_map_fits).exists() and 4 in iZs)

if _z4_can_correct:
    print(f'Loading metrology: {height_map_fits}')
    metrology = make_metrology_table(height_map_fits)
    print(f'  {len(metrology)} metrology points across '
          f'{len(set(metrology["det"]))} sensors')

    from lsst.obs.lsst import LsstCam       # local import (LSST stack only)
    camera = LsstCam.getCamera()

    factor = (height_to_z4_factor
              if height_to_z4_factor is not None
              else HEIGHT_TO_Z4_UM_PER_MM)
    interp = get_height_interpolator(metrology)

    # Per-donut focal-plane (mm) coords -> height -> Z4 contribution
    fpx, fpy = compute_fp_coords(donut_df, camera)
    Z4hgt = height_to_z4(interp(fpx, fpy), factor=factor)
    print(f'Z4hgt   per donut (data side): mean={float(np.nanmean(Z4hgt)):.4f} μm  '
          f'std={float(np.nanstd(Z4hgt)):.4f} μm')

    # Intrinsic side: maybe transposed
    if intrinsic_transpose_bug:
        det_names = np.asarray(donut_df['detector']).astype(str)
        fpx_t, fpy_t = transpose_around_ccd_centers(fpx, fpy, det_names, camera)
        Z4hgt_intrinsic = height_to_z4(interp(fpx_t, fpy_t), factor=factor)
        print(f'Z4hgt_t per donut (intrinsic, x<->y per CCD): '
              f'mean={float(np.nanmean(Z4hgt_intrinsic)):.4f} μm  '
              f'std={float(np.nanstd(Z4hgt_intrinsic)):.4f} μm')
    else:
        Z4hgt_intrinsic = Z4hgt
        print('Using un-transposed Z4hgt for the intrinsic '
              '(intrinsic_transpose_bug=False)')

    data_offset = {4: Z4hgt}
    intrinsic_offset = {4: Z4hgt_intrinsic}
    print('Z4 corrections will be applied inside build_measured_intrinsic '
          '(measured Z4 -= Z4hgt; intrinsic Z4 -= '
          f'{"Z4hgt_transpose" if intrinsic_transpose_bug else "Z4hgt"}).')
else:
    if not _ccd_height_ok:
        print('Skipping Z4 height correction: ccd_height module unavailable')
    elif not height_map_fits:
        print('Skipping Z4 height correction: height_map_fits is None')
    elif not Path(height_map_fits).exists():
        print(f'Skipping Z4 height correction: {height_map_fits} not found')
    elif 4 not in iZs:
        print('Skipping Z4 height correction: Z4 not in iZs')

<a id='analysis'></a>
## 6. DZ Fit + Iteration

For each visit, fit the chosen `(k, j)` DZ modes to
`data − tabulated_intrinsic`.  Subtract only the DZ contribution from
the data (the intrinsic content is *kept* — it is what we are trying to
measure).  Median over visits → iter-1 measured intrinsic.  Replace the
tabulated intrinsic with the iter-1 grid (linearly interpolated at each
donut's field position) and repeat → iter-2.

The driver `build_measured_intrinsic` performs both iterations and
returns all the intermediate maps + DZ fit tables.  When the Z4
height-correction was computed in §5, it is passed in as
`data_offset` / `intrinsic_offset`, so every map and fit downstream
uses Z4_measured − Z4hgt and Z4_intrinsic − Z4hgt_transpose.

In [ ]:
result = build_measured_intrinsic(
    donut_df, visits_kept, coord_sys, iZs, removal_spec,
    n_iter=n_iter,
    n_bins=n_bins,
    fp_radius_basis=fp_radius_basis,
    fp_radius_grid=fp_radius_grid,
    min_donuts=min_donuts,
    bad_fit_threshold=bad_fit_threshold,
    data_offset=data_offset,
    intrinsic_offset=intrinsic_offset,
)
print(f'\nIterations completed: {len(result["iter_results"])}')

for it_idx, it in enumerate(result['iter_results']):
    n_total = len(it['fit_rows'])
    n_bad = sum(1 for r in it['fit_rows'] if r.get('bad_fit', False))
    n_good = n_total - n_bad
    print(f'  iter {it_idx + 1}: {n_good}/{n_total} visits passed '
          f'(min_donuts >= {min_donuts}, |coeff| <= {bad_fit_threshold} μm)')

<a id='plots'></a>
## 7. Comparison Plots — 4-panel per pupil j (2x2)

For each pupil Noll j (in `iZs`):

| Panel 1 | Panel 2 | Panel 3 | Panel 4 |
|---|---|---|---|
| Original median (raw `zk`) | Iter-1 measured | Iter-2 measured | Tabulated intrinsic |

Panel 1 uses its own 5–95 % colour scale; panels 2–4 share a single
5–95 % range pooled across those three.  When the Z4 correction is on,
the j=4 page shows `Z4 − Z4hgt` and `Z4_intrinsic − Z4hgt_transpose`.

In [ ]:
def plot_4panel_for_j(j, original_grid, iter1_grid, iter2_grid,
                       tabulated_grid, xbins, ybins,
                       coord_sys, plo=5.0, phi=95.0):
    """4-panel comparison page for one pupil j on a 2x2 grid.

    Color scales:
      * Panel 1 (Original median) uses its own 5-95 % range — the raw
        zk often has a much larger dynamic range than the intrinsic.
      * Panels 2-4 (Iter-1, Iter-2, Tabulated) share a single 5-95 %
        range pooled over those three panels so they are directly
        comparable.
    """
    panels = [
        ('Original median (raw zk)', original_grid),
        ('Iter-1 measured', iter1_grid),
        ('Iter-2 measured', iter2_grid),
        ('Tabulated intrinsic', tabulated_grid),
    ]
    # Panel 1: own range
    if original_grid is not None and np.any(np.isfinite(original_grid)):
        vmin0, vmax0 = np.nanpercentile(original_grid, [plo, phi])
    else:
        vmin0, vmax0 = -1.0, 1.0
    # Panels 2-4: shared range
    intrinsic_grids = [iter1_grid, iter2_grid, tabulated_grid]
    vmin_i, vmax_i = common_color_scale(intrinsic_grids, plo=plo, phi=phi)

    fig, axes = plt.subplots(2, 2, figsize=(13, 11),
                             layout='constrained')
    axes = axes.ravel()
    for idx, (ax, (title, grid)) in enumerate(zip(axes, panels)):
        if grid is None or not np.any(np.isfinite(grid)):
            ax.set_visible(False)
            continue
        if idx == 0:
            vmin, vmax = vmin0, vmax0
            scale_note = f'5-95% own = [{vmin0:.3f}, {vmax0:.3f}] μm'
        else:
            vmin, vmax = vmin_i, vmax_i
            scale_note = f'shared 5-95% = [{vmin_i:.3f}, {vmax_i:.3f}] μm'
        im = ax.imshow(grid.T, origin='lower',
                       extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
                       cmap='viridis', interpolation='none', aspect='equal',
                       vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.8, label='μm')
        ax.set_xlabel(f'thy_{coord_sys} [deg]')
        ax.set_ylabel(f'thx_{coord_sys} [deg]')
        ax.set_title(f'{title}\n({scale_note})', fontsize=10)
    fig.suptitle(
        f'Pupil Z{j}  ({coord_sys})  — Original uses own 5-95%; '
        f'Iter-1 / Iter-2 / Tabulated share a single 5-95%',
        fontsize=12)
    return fig


# Build & display the comparison figures
comparison_figs = []
xbins, ybins = result['xbins'], result['ybins']
iter1_grids = result['iter_results'][0]['measured_grid']
iter2_grids = (result['iter_results'][1]['measured_grid']
               if len(result['iter_results']) >= 2
               else result['iter_results'][-1]['measured_grid'])

for j in iZs:
    if j not in by_pupil:
        # not in removal spec; skip (no DZ subtraction defined)
        continue
    fig = plot_4panel_for_j(
        j,
        result['original_median'].get(j),
        iter1_grids.get(j),
        iter2_grids.get(j),
        result['tabulated_median'].get(j),
        xbins, ybins, coord_sys)
    comparison_figs.append(fig)
    plt.show()

print(f'Built {len(comparison_figs)} per-pupil-j comparison figures')

<a id='tracking'></a>
## 8. Per-DZ-term Tracking — Iter-1 vs Iter-2

Two PDFs, both 4×4 grids per page over the (k, j) modes in the removal
spec:

* **`measured_intrinsic_tracking.pdf`** — fit coefficient vs visit, with
  thin error bars from the fit `_err` column.  Visits flagged as bad
  (`min_donuts < 500` or any `|coeff| > 2 μm`) are excluded so the plot
  shows exactly what contributed to the median maps.
* **`measured_intrinsic_tracking_robust_rms.pdf`** — per-coefficient
  robust RMS vs visit.  The robust RMS is the standard error from the
  Huber RLM fit (`dz_z{j}_c{k}_err`) — a MAD-equivalent dispersion
  estimator.

Iter-1 = filled blue circles, iter-2 = open red squares.

In [ ]:
def _good_indices(fit_rows):
    """Indices of fit_rows that passed the quality cut (bad_fit == False)."""
    return [i for i, r in enumerate(fit_rows) if not r.get('bad_fit', False)]


def _series(rows, idx, k, j, dz_prefix='dz', err=False):
    suffix = '_err' if err else ''
    col = f'{dz_prefix}_z{j}_c{k}{suffix}'
    return np.array([float(rows[i].get(col, np.nan)) for i in idx])


def _add_day_obs_dividers(ax, day_obs_seq, label_top=False):
    """Vertical dotted lines + tiny rotated day_obs labels at day boundaries."""
    if len(day_obs_seq) < 2:
        return
    if label_top:
        # Force renderer to compute the y-range before placing labels
        ax.figure.canvas.draw_idle()
    ymax = ax.get_ylim()[1]
    for i in range(1, len(day_obs_seq)):
        if day_obs_seq[i] != day_obs_seq[i - 1]:
            ax.axvline(i - 0.5, color='black', linewidth=0.5,
                       linestyle=':', alpha=0.45, zorder=0)
            if label_top:
                ax.text(i - 0.5, ymax, str(day_obs_seq[i]),
                        fontsize=5, rotation=90, va='top', ha='right',
                        alpha=0.7, zorder=1)


def per_term_pages(fit_rows_iter1, fit_rows_iter2, by_pupil,
                   coord_sys, dz_prefix='dz', plots_per_page=16, ncols=4,
                   skip_bad=True):
    """One panel per (k, j) — fit coefficient vs visit, both iterations.

    Bad-flagged visits are excluded when skip_bad=True (default).  Vertical
    dotted lines mark day_obs boundaries; the first panel of every page also
    carries small rotated day_obs labels.
    """
    pairs = sorted(((k, j) for j, ks in by_pupil.items() for k in ks),
                   key=lambda kj: (kj[0], kj[1]))
    idx1 = _good_indices(fit_rows_iter1) if skip_bad else list(
        range(len(fit_rows_iter1)))
    idx2 = _good_indices(fit_rows_iter2) if skip_bad else list(
        range(len(fit_rows_iter2)))

    dobs1 = [int(fit_rows_iter1[i]['day_obs']) for i in idx1]

    n = len(pairs)
    nrows = plots_per_page // ncols
    pages = []
    for page_start in range(0, n, plots_per_page):
        page = pairs[page_start:page_start + plots_per_page]
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(4.0 * ncols, 3.0 * nrows),
                                 layout='constrained')
        axes = np.atleast_1d(axes).ravel()
        for idx, (k, j) in enumerate(page):
            ax = axes[idx]
            v1 = _series(fit_rows_iter1, idx1, k, j, dz_prefix)
            v2 = _series(fit_rows_iter2, idx2, k, j, dz_prefix)
            e1 = _series(fit_rows_iter1, idx1, k, j, dz_prefix, err=True)
            e2 = _series(fit_rows_iter2, idx2, k, j, dz_prefix, err=True)
            ax.errorbar(np.arange(len(v1)), v1, yerr=e1, fmt='o',
                        color='tab:blue', markersize=4, alpha=0.85,
                        elinewidth=0.5, capsize=0, label='iter 1')
            ax.errorbar(np.arange(len(v2)), v2, yerr=e2, fmt='s',
                        color='tab:red', markersize=4, alpha=0.85,
                        elinewidth=0.5, capsize=0, mfc='none', label='iter 2')
            ax.axhline(0, color='gray', lw=0.5, alpha=0.5)
            ax.set_title(f'(k={k}, j={j})', fontsize=10)
            ax.tick_params(labelsize=7)
            if idx == 0:
                ax.legend(fontsize=7, loc='best')
            _add_day_obs_dividers(ax, dobs1, label_top=(idx == 0))
        for idx in range(len(page), len(axes)):
            axes[idx].set_visible(False)
        page_idx = page_start // plots_per_page + 1
        n_pages = int(np.ceil(n / plots_per_page))
        n_skipped = (len(fit_rows_iter1) - len(idx1)
                     if skip_bad else 0)
        suffix = (f'  (good visits only, dropped {n_skipped} bad-flagged)'
                  if skip_bad else '')
        fig.suptitle(
            f'DZ fit coefficients vs visit  '
            f'({page_idx}/{n_pages}, {coord_sys}){suffix}',
            fontsize=12)
        pages.append(fig)
    return pages


def per_term_robust_rms_pages(fit_rows_iter1, fit_rows_iter2, by_pupil,
                              coord_sys, dz_prefix='dz',
                              plots_per_page=16, ncols=4,
                              skip_bad=True):
    """One panel per (k, j) — robust RMS of the fit coefficient vs visit."""
    pairs = sorted(((k, j) for j, ks in by_pupil.items() for k in ks),
                   key=lambda kj: (kj[0], kj[1]))
    idx1 = _good_indices(fit_rows_iter1) if skip_bad else list(
        range(len(fit_rows_iter1)))
    idx2 = _good_indices(fit_rows_iter2) if skip_bad else list(
        range(len(fit_rows_iter2)))

    dobs1 = [int(fit_rows_iter1[i]['day_obs']) for i in idx1]

    n = len(pairs)
    nrows = plots_per_page // ncols
    pages = []
    for page_start in range(0, n, plots_per_page):
        page = pairs[page_start:page_start + plots_per_page]
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(4.0 * ncols, 3.0 * nrows),
                                 layout='constrained')
        axes = np.atleast_1d(axes).ravel()
        for idx, (k, j) in enumerate(page):
            ax = axes[idx]
            e1 = _series(fit_rows_iter1, idx1, k, j, dz_prefix, err=True)
            e2 = _series(fit_rows_iter2, idx2, k, j, dz_prefix, err=True)
            ax.plot(np.arange(len(e1)), e1, 'o',
                    color='tab:blue', markersize=4, alpha=0.85,
                    label='iter 1')
            ax.plot(np.arange(len(e2)), e2, 's',
                    color='tab:red', markersize=4, alpha=0.85,
                    mfc='none', label='iter 2')
            ax.axhline(0, color='gray', lw=0.3, alpha=0.5)
            ax.set_title(f'(k={k}, j={j})  σ_robust', fontsize=10)
            ax.set_ylim(bottom=0)
            ax.tick_params(labelsize=7)
            if idx == 0:
                ax.legend(fontsize=7, loc='best')
            _add_day_obs_dividers(ax, dobs1, label_top=(idx == 0))
        for idx in range(len(page), len(axes)):
            axes[idx].set_visible(False)
        page_idx = page_start // plots_per_page + 1
        n_pages = int(np.ceil(n / plots_per_page))
        suffix = '  (good visits only)' if skip_bad else ''
        fig.suptitle(
            f'Per-coefficient robust RMS (RLM bse)  '
            f'({page_idx}/{n_pages}, {coord_sys}){suffix}',
            fontsize=12)
        pages.append(fig)
    return pages


tracking_figs = []
tracking_err_figs = []
if len(result['iter_results']) >= 2:
    rows1 = result['iter_results'][0]['fit_rows']
    rows2 = result['iter_results'][1]['fit_rows']
    n1_bad = sum(1 for r in rows1 if r.get('bad_fit', False))
    n2_bad = sum(1 for r in rows2 if r.get('bad_fit', False))
    print(f'Tracking plot: dropping {n1_bad} bad-flagged iter-1 visits '
          f'and {n2_bad} from iter-2.')

    tracking_figs = per_term_pages(rows1, rows2, by_pupil, coord_sys,
                                   skip_bad=True)
    for fig in tracking_figs:
        plt.show()
    print(f'Built {len(tracking_figs)} per-DZ-term tracking pages')

    tracking_err_figs = per_term_robust_rms_pages(
        rows1, rows2, by_pupil, coord_sys, skip_bad=True)
    for fig in tracking_err_figs:
        plt.show()
    print(f'Built {len(tracking_err_figs)} per-DZ-term robust-RMS pages')
else:
    print('Skipping tracking pages: only one iteration available')

<a id='validation'></a>
## 9. Per-visit Residual Validation

For each (good) visit, compute the per-donut residual

   residual = zk_data − (measured_intrinsic_iter2 + DZ_fit_iter2)

evaluated at each donut's (thx, thy).  The Z4 corrections from §5 are
already baked into `zk_data` and the iter-2 grid, so the residual is
height-aware.

Pages are **streamed directly** to `measured_intrinsic_validation.pdf`
in this cell (one figure in memory at a time, closed after savefig) so
the validation step doesn't accumulate hundreds of figure objects.

Page order:

1. Per-visit histograms (always) — 4×4 grid of 1-D residual histograms
   for pupil j = 4..19.
2. Per-visit maps (optional, slow) — `make_residual_maps` toggle.
3. Marker legend page.
4. Pooled σ vs visit, pooled σ_MAD vs visit.
5. Per-j σ vs visit and σ_MAD vs visit pages for j in
   `per_zernike_sigma_j` (default 4..8).

Marker shape encodes rotator group and color encodes elevation group;
day_obs boundaries are marked with vertical dotted lines + small rotated
day_obs labels at the top.

In [ ]:
def compute_validation_residual(donut_df, result, coord_sys, iZs):
    """Per-donut residual after subtracting iter-2 intrinsic AND iter-2 DZ fit."""
    final_iter = result['iter_results'][-1]
    wfd_sub = final_iter['wfd_subtracted']
    measured_grid = final_iter['measured_grid']
    xcent, ycent = result['xcent'], result['ycent']
    thx = np.rad2deg(np.asarray(donut_df[f'thx_{coord_sys}'], dtype=float))
    thy = np.rad2deg(np.asarray(donut_df[f'thy_{coord_sys}'], dtype=float))
    intrinsic_at_donut = interpolate_grid_at_donuts(
        measured_grid, xcent, ycent, thx, thy, iZs)
    return wfd_sub - intrinsic_at_donut


# ---- Per-visit summary helpers (elev/rot grouping + day_obs dividers) ----

ELEV_CENTERS = (30, 40, 50, 60, 70)
ELEV_HALFWIDTH = 5
ROT_CENTERS    = (-60, -45, -30, -15, 0, 15, 30, 45, 60)

ELEV_COLORS = {
    30: 'tab:blue', 40: 'tab:green', 50: 'tab:orange',
    60: 'tab:red',  70: 'tab:purple',
}
ROT_MARKERS = {
    -60: 'D', -45: 'v', -30: '<', -15: 'o', 0: 's',
     15: '>',  30: '^',  45: 'P', 60: '*',
}


def _alt_to_deg(alt_value):
    a = float(alt_value)
    return np.rad2deg(a) if abs(a) < 2.0 * np.pi + 1e-3 else a


def _elev_group(alt_deg):
    if not np.isfinite(alt_deg):
        return None
    for c in ELEV_CENTERS:
        if (c - ELEV_HALFWIDTH) <= alt_deg < (c + ELEV_HALFWIDTH):
            return c
    return None


def _rot_group(rot_deg):
    if rot_deg is None or not np.isfinite(rot_deg):
        return None
    centers = np.array(ROT_CENTERS, dtype=float)
    return int(centers[int(np.argmin(np.abs(centers - rot_deg)))])


def _build_visit_lookup(visits_table):
    out = {}
    for v in visits_table:
        d = int(v['day_obs'])
        s = int(v['seq_num'])
        alt_deg = _alt_to_deg(v['alt']) if 'alt' in visits_table.colnames \
            else np.nan
        rot_deg = float(v['rotator_angle']) if 'rotator_angle' in \
            visits_table.colnames else np.nan
        out[(d, s)] = {'alt_deg': alt_deg, 'rot_deg': rot_deg}
    return out


def _markers_legend_page():
    fig = plt.figure(figsize=(11, 8.5))
    ax = fig.add_subplot(111)
    ax.set_axis_off()
    ax.set_title('Per-visit summary marker key', fontsize=14, pad=20)

    handles = []
    labels = []
    for c, m in ROT_MARKERS.items():
        handles.append(plt.Line2D([], [], linestyle='', marker=m,
                                  markersize=10, color='gray',
                                  mfc='none'))
        labels.append(f'rot ≈ {c:+d}°')
    leg1 = ax.legend(handles, labels,
                     loc='upper left', bbox_to_anchor=(0.05, 0.95),
                     title='rotator_angle (marker shape)',
                     fontsize=10, title_fontsize=11, frameon=True)
    ax.add_artist(leg1)

    handles2 = []
    labels2 = []
    for c, color in ELEV_COLORS.items():
        handles2.append(plt.Line2D([], [], linestyle='', marker='o',
                                   markersize=10, color=color))
        labels2.append(f'alt ∈ [{c - ELEV_HALFWIDTH}, '
                       f'{c + ELEV_HALFWIDTH}) → {c}°')
    ax.legend(handles2, labels2,
              loc='upper right', bbox_to_anchor=(0.95, 0.95),
              title='elevation (marker color)',
              fontsize=10, title_fontsize=11, frameon=True)

    ax.text(0.5, 0.05,
            'Visits outside both grouping ranges are drawn as gray X.',
            ha='center', va='bottom', transform=ax.transAxes,
            fontsize=10, color='gray')
    return fig


def _make_sigma_time_plot(visits_x, yvals, dobs_seq,
                          elev_groups, rot_groups,
                          ylabel, title):
    fig, ax = plt.subplots(figsize=(14, 5), layout='constrained')
    for i in range(len(visits_x)):
        if not np.isfinite(yvals[i]):
            continue
        eg = elev_groups[i]
        rg = rot_groups[i]
        if eg is None or rg is None:
            ax.plot(visits_x[i], yvals[i], 'x', color='gray',
                    markersize=6, alpha=0.7)
        else:
            ax.plot(visits_x[i], yvals[i],
                    marker=ROT_MARKERS[rg], color=ELEV_COLORS[eg],
                    markersize=7, mfc=ELEV_COLORS[eg], alpha=0.9,
                    linestyle='')
    ax.set_xlabel('visit index')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)
    _add_day_obs_dividers(ax, dobs_seq, label_top=True)
    return fig


def _sigma_pair(res):
    res = res[np.isfinite(res)]
    if len(res) < 5:
        return np.nan, np.nan
    sigma = float(np.std(res))
    mad = float(np.median(np.abs(res - np.median(res))))
    return sigma, 1.4826 * mad


def per_visit_validation_to_pdf(donut_df, result, coord_sys, iZs,
                                output_pdf_path,
                                j_range, hist_range_um=(-1.0, 1.0),
                                n_hist_bins=60,
                                map_n_bins=16, map_fp_radius=1.8,
                                make_maps=False, ncols=4,
                                visits_table=None,
                                per_j_sigma_range=None,
                                show_first=True):
    """Stream per-visit residual histogram (and optional map) pages plus the
    final summary block directly to ``output_pdf_path``.  Each figure is
    closed (`plt.close`) immediately after `pdf.savefig`, so memory stays
    bounded at one page at a time even for large multi-chunk runs.

    Returns the number of pages written.
    """
    from scipy.stats import binned_statistic_2d

    # Compute per-donut residuals once (n_donuts, n_zern); drop after use.
    residuals = compute_validation_residual(donut_df, result, coord_sys, iZs)
    iZidx = {iZ: i for i, iZ in enumerate(iZs)}
    j_list = [j for j in j_range if j in iZidx]
    n = len(j_list)
    nrows = (n + ncols - 1) // ncols

    final_iter = result['iter_results'][-1]
    good_rows = [r for r in final_iter['fit_rows']
                 if not r.get('bad_fit', False)]

    dobs_full = np.asarray(donut_df['day_obs'])
    snum_full = np.asarray(donut_df['seq_num'])

    # Per-j summary range
    j_per = [j for j in (per_j_sigma_range
                         if per_j_sigma_range is not None
                         else range(4, 9))
             if j in iZidx]
    j_pool_idx = [iZidx[j] for j in j_list]

    # Pre-allocate per-visit summary collectors (small)
    visit_meta = (_build_visit_lookup(visits_table)
                  if visits_table is not None else {})
    sigmas_pool, sigma_mads_pool = [], []
    sigma_per_j     = {j: [] for j in j_per}
    sigma_mad_per_j = {j: [] for j in j_per}
    dobs_seq, elev_groups, rot_groups = [], [], []

    Path(output_pdf_path).parent.mkdir(parents=True, exist_ok=True)
    n_pages = 0
    print(f'Streaming validation PDF -> {output_pdf_path}')
    print(f'  {len(good_rows)} good visits × '
          f'{2 if make_maps else 1} page(s)/visit '
          f'(j = {j_list[0]}..{j_list[-1]}, hist range {hist_range_um})')

    with PdfPages(output_pdf_path) as pdf:
        for r_idx, r in enumerate(good_rows):
            d = int(r['day_obs'])
            s = int(r['seq_num'])
            mask = (dobs_full == d) & (snum_full == s)
            if not np.any(mask):
                continue
            n_donuts = int(mask.sum())

            # ---- Histogram page ----
            fig, axes = plt.subplots(nrows, ncols,
                                     figsize=(3.5 * ncols, 2.8 * nrows),
                                     layout='constrained')
            axes = np.atleast_1d(axes).ravel()
            for idx, j in enumerate(j_list):
                ax = axes[idx]
                res = residuals[mask, iZidx[j]]
                res = res[np.isfinite(res)]
                if len(res) < 2:
                    ax.set_visible(False)
                    continue
                ax.hist(res, bins=n_hist_bins, range=hist_range_um,
                        edgecolor='black', linewidth=0.3, alpha=0.75)
                sigma = float(np.std(res))
                mad = float(np.median(np.abs(res - np.median(res))))
                sigma_mad_val = 1.4826 * mad
                n_in = int(np.sum((res >= hist_range_um[0])
                                  & (res <= hist_range_um[1])))
                ax.text(0.97, 0.95,
                        f'σ={sigma:.3f}\nσ_MAD={sigma_mad_val:.3f}\n'
                        f'{n_in}/{len(res)} in range',
                        transform=ax.transAxes, ha='right', va='top',
                        fontsize=8,
                        bbox=dict(boxstyle='round', facecolor='wheat',
                                  alpha=0.5))
                ax.axvline(0, color='gray', lw=0.5, alpha=0.5)
                ax.set_title(f'j={j}', fontsize=10)
                ax.set_xlabel('residual [μm]', fontsize=8)
                ax.tick_params(labelsize=7)
                ax.set_xlim(*hist_range_um)
            for idx in range(len(j_list), len(axes)):
                axes[idx].set_visible(False)
            fig.suptitle(
                f'Visit ({d}, {s})  [{r_idx + 1}/{len(good_rows)}]  — '
                f'residual histograms (data − iter2_intrinsic − iter2_DZ, '
                f'n_donuts={n_donuts})',
                fontsize=12)
            if show_first and r_idx == 0:
                plt.show()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            n_pages += 1

            # ---- Optional residual map page ----
            if make_maps:
                xbins_m = np.linspace(-map_fp_radius, map_fp_radius,
                                      map_n_bins + 1)
                ybins_m = np.linspace(-map_fp_radius, map_fp_radius,
                                      map_n_bins + 1)
                thx_v = np.rad2deg(np.asarray(donut_df[f'thx_{coord_sys}'],
                                              dtype=float))[mask]
                thy_v = np.rad2deg(np.asarray(donut_df[f'thy_{coord_sys}'],
                                              dtype=float))[mask]

                fig, axes = plt.subplots(nrows, ncols,
                                         figsize=(3.5 * ncols, 3.2 * nrows),
                                         layout='constrained')
                axes = np.atleast_1d(axes).ravel()
                for idx, j in enumerate(j_list):
                    ax = axes[idx]
                    res = residuals[mask, iZidx[j]]
                    if not np.any(np.isfinite(res)):
                        ax.set_visible(False)
                        continue
                    stat_val, _, _, _ = binned_statistic_2d(
                        thy_v, thx_v, res, statistic='median',
                        bins=[xbins_m, ybins_m])
                    vmin, vmax = np.nanpercentile(res, [5, 95])
                    im = ax.imshow(
                        stat_val.T, origin='lower',
                        extent=[xbins_m[0], xbins_m[-1],
                                ybins_m[0], ybins_m[-1]],
                        cmap='RdBu_r', interpolation='none',
                        aspect='equal',
                        vmin=vmin, vmax=vmax)
                    plt.colorbar(im, ax=ax, shrink=0.65, label='μm')
                    ax.set_title(f'j={j}', fontsize=10)
                    ax.set_xlabel(f'thy_{coord_sys} [deg]', fontsize=8)
                    ax.set_ylabel(f'thx_{coord_sys} [deg]', fontsize=8)
                    ax.tick_params(labelsize=7)
                for idx in range(len(j_list), len(axes)):
                    axes[idx].set_visible(False)
                fig.suptitle(
                    f'Visit ({d}, {s})  [{r_idx + 1}/{len(good_rows)}]  — '
                    f'residual maps ({map_n_bins}×{map_n_bins} bins, '
                    f'n_donuts={n_donuts})',
                    fontsize=12)
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                n_pages += 1

            # ---- Per-visit summary collection ----
            res_pool = residuals[mask][:, j_pool_idx].ravel()
            sp, sm = _sigma_pair(res_pool)
            if not np.isfinite(sp):
                continue
            sigmas_pool.append(sp)
            sigma_mads_pool.append(sm)
            for j in j_per:
                res_j = residuals[mask, iZidx[j]]
                sj, smj = _sigma_pair(res_j)
                sigma_per_j[j].append(sj)
                sigma_mad_per_j[j].append(smj)
            dobs_seq.append(d)
            meta = visit_meta.get((d, s),
                                  {'alt_deg': np.nan, 'rot_deg': np.nan})
            elev_groups.append(_elev_group(meta['alt_deg']))
            rot_groups.append(_rot_group(meta['rot_deg']))

            if (r_idx + 1) % 10 == 0:
                print(f'    ...wrote {r_idx + 1}/{len(good_rows)} visits')

        # Drop the heavy residuals array now that all per-visit work is done.
        del residuals

        # ---- Summary block (legend + pooled + per-j) ----
        if visits_table is not None and len(sigmas_pool) > 0:
            sigmas_pool_arr = np.array(sigmas_pool)
            sigma_mads_pool_arr = np.array(sigma_mads_pool)
            visits_x = np.arange(len(sigmas_pool_arr))

            pool_lo, pool_hi = j_list[0], j_list[-1]

            for fig in [_markers_legend_page()]:
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                n_pages += 1

            for ylabel, yvals, title in [
                ('σ (std) [μm]', sigmas_pool_arr,
                 f'Pooled per-visit residual σ vs visit  '
                 f'({coord_sys}; j={pool_lo}..{pool_hi}, '
                 f'n_visits={len(sigmas_pool_arr)})'),
                ('σ_MAD = 1.4826·MAD [μm]', sigma_mads_pool_arr,
                 f'Pooled per-visit residual σ_MAD vs visit  '
                 f'({coord_sys}; j={pool_lo}..{pool_hi})'),
            ]:
                fig = _make_sigma_time_plot(
                    visits_x, yvals, dobs_seq, elev_groups, rot_groups,
                    ylabel=ylabel, title=title)
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                n_pages += 1

            for j in j_per:
                fig = _make_sigma_time_plot(
                    visits_x, np.array(sigma_per_j[j]), dobs_seq,
                    elev_groups, rot_groups,
                    ylabel='σ (std) [μm]',
                    title=f'Per-visit residual σ vs visit  '
                          f'({coord_sys}; j={j})')
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                n_pages += 1

            for j in j_per:
                fig = _make_sigma_time_plot(
                    visits_x, np.array(sigma_mad_per_j[j]), dobs_seq,
                    elev_groups, rot_groups,
                    ylabel='σ_MAD = 1.4826·MAD [μm]',
                    title=f'Per-visit residual σ_MAD vs visit  '
                          f'({coord_sys}; j={j})')
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                n_pages += 1

    print(f'Wrote {n_pages} pages.')
    return n_pages


validation_pdf_path = None
if len(result['iter_results']) >= 2:
    if output_dir is None:
        output_dir = output_dir_default(
            donut_parquet, coord_sys, day_obs_min, day_obs_max,
            seq_num_min, seq_num_max,
            alt_min_deg, alt_max_deg,
            rot_min_deg, rot_max_deg)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    validation_pdf_path = str(output_dir / 'measured_intrinsic_validation.pdf')

    n_val_pages = per_visit_validation_to_pdf(
        donut_df, result, coord_sys, iZs,
        output_pdf_path=validation_pdf_path,
        j_range=residual_j_range,
        hist_range_um=residual_hist_range,
        n_hist_bins=residual_n_hist_bins,
        map_n_bins=residual_map_n_bins,
        map_fp_radius=residual_map_fp_radius,
        make_maps=make_residual_maps,
        visits_table=visits_kept,
        per_j_sigma_range=per_zernike_sigma_j,
        show_first=True)
else:
    print('Skipping validation pages: only one iteration available')

<a id='output'></a>
## 10. Output Tables

In [ ]:
if output_dir is None:
    output_dir = output_dir_default(
        donut_parquet, coord_sys, day_obs_min, day_obs_max,
        seq_num_min, seq_num_max,
        alt_min_deg, alt_max_deg,
        rot_min_deg, rot_max_deg)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {output_dir}')

# 1. Iter-2 (final) DZ fit table — per-visit coefficients keyed dz_z{j}_c{k}
final_iter = result['iter_results'][-1]
dz_fits_path = output_dir / 'measured_intrinsic_dz_fits.parquet'
save_dz_fits(final_iter['fit_rows'], dz_fits_path)

# 2. Measured-intrinsic grid table (single-frame in `coord_sys`).
xcent, ycent = result['xcent'], result['ycent']
intrinsic_table = assemble_intrinsic_table(
    grid=iter2_grids, iZs=iZs, xcent=xcent, ycent=ycent,
    coord_sys_grid=coord_sys, alt_coord_xform=None)
print(f'Intrinsic grid table: {len(intrinsic_table)} non-empty bins '
      f'in {coord_sys}')

intrinsic_path = output_dir / f'measured_intrinsic_grid.{output_format}'
if output_format == 'parquet':
    intrinsic_table.write(str(intrinsic_path), format='parquet',
                          overwrite=True)
elif output_format == 'fits':
    intrinsic_table.write(str(intrinsic_path), format='fits',
                          overwrite=True)
elif output_format == 'hdf5':
    intrinsic_table.write(str(intrinsic_path), format='hdf5', path='grid',
                          overwrite=True, append=False)
else:
    raise ValueError(f'Unknown output_format: {output_format}')
print(f'Saved measured intrinsic grid: {intrinsic_path}')

# 3. PDFs:
#    - measured_intrinsic_comparison.pdf            : 4-panel per pupil j
#    - measured_intrinsic_tracking.pdf              : per-(k,j) coeff vs visit
#    - measured_intrinsic_tracking_robust_rms.pdf   : per-(k,j) robust σ
#    - measured_intrinsic_validation.pdf            : per-visit residuals
#      (already streamed by section 9 — see validation_pdf_path)
comp_pdf = output_dir / 'measured_intrinsic_comparison.pdf'
with PdfPages(comp_pdf) as pdf:
    for fig in comparison_figs:
        pdf.savefig(fig, bbox_inches='tight')
print(f'Saved comparison PDF:  {comp_pdf}')

if tracking_figs:
    track_pdf = output_dir / 'measured_intrinsic_tracking.pdf'
    with PdfPages(track_pdf) as pdf:
        for fig in tracking_figs:
            pdf.savefig(fig, bbox_inches='tight')
    print(f'Saved tracking PDF:    {track_pdf}')

if tracking_err_figs:
    err_pdf = output_dir / 'measured_intrinsic_tracking_robust_rms.pdf'
    with PdfPages(err_pdf) as pdf:
        for fig in tracking_err_figs:
            pdf.savefig(fig, bbox_inches='tight')
    print(f'Saved robust-RMS PDF:  {err_pdf}')

if validation_pdf_path is not None:
    print(f'Validation PDF (streamed in §9): {validation_pdf_path}')

<a id='format'></a>
## 11. Output Format Options — Rubin DM datasets

The "measured intrinsic" map is a **per-pupil-j focal-plane grid of
median Zernike values** — conceptually parallel to the existing batoid
intrinsic table (`zk_intrinsic_*` columns produced upstream by
`aggregateAOSVisitTable`), but data-derived.  Realistic choices:

| Option | Pros | Cons |
|---|---|---|
| **Native parquet** (default) | Matches every other AOS pipeline output; loadable by `astropy.QTable.read` and `pandas.read_parquet`; round-trips list columns natively. | Not registered with the Butler — passed by path. |
| **FITS BinTable** | DM-friendly, viewable in DS9/topcat. | Slightly heavier; list columns become variable-length arrays. |
| **HDF5 QTable** | Multi-table file possible. | Less standard for AOS outputs going forward. |
| **Custom Butler dataset type** | Full provenance, `butler.get(...)`. | Requires an RFC + StorageClass + dimension records. |

Iterating with parquet for now; once the iteration logic is stable a
Butler dataset type is a natural next step.